# 3-Stage 평가서버 호환 베이스라인 — 학습

공개 예제 각 5건으로 모델을 학습하고 실제 참가자 제출구조와 동일한 경로에 체크포인트를 저장합니다.

In [ ]:
%pip install -r requirements.txt

## 1. 라이브러리·모델 구조·학습함수

In [ ]:
from pathlib import Path
import itertools, json, os, random
import cv2, numpy as np, pandas as pd, torch
from torch import nn
from torchvision.models.video import mvit_v2_s
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_320_fpn,
)


In [ ]:
ROOT=Path.cwd(); DATA=ROOT/'data'; MODEL=ROOT/'model'
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(20260825); random.seed(20260825)

# Stage1: 실제 제출 점수(0.316)가 팀 베이스라인 MViT(0.53397)보다 낮게 나와서 원복함 -
# 핸드크래프트 피처+로지스틱회귀는 합성 재녹화에만 과적합된 것으로 보임(실제 재녹화
# 데이터로 검증 전까지는 미사용). 아래 fit_stage1()은 MViT, 실험용 핸드크래프트 버전은
# fit_stage1_experimental_handcrafted()로 따로 둠(기본 실행 대상 아님).
SIZE=224
S1_MEAN=torch.tensor([0.45,0.45,0.45])[:,None,None,None]
S1_STD=torch.tensor([0.225,0.225,0.225])[:,None,None,None]
FEATURE_DIM=5  # 실험용 핸드크래프트 피처 차원

# Stage2: 충돌/진입 시점 (COCO 사전학습 탐지기 + 학습형 재정렬기)
VEHICLE_CLASSES={'car','motorcycle','bus','truck'}
SCORE_THR=0.2  # 0.5->0.2: 358영상/16248프레임 캐시 검증(mean IoU 0.378->0.442, 히트율 45.2%->53.1%, 0.2 밑은 수확체감)
RERANK_FEATURES=['cx','cy','bw','bh','score','aspect']

# Stage3: 가감속/조향 (optical flow 휴리스틱, 학습 없음 - 임계값만 보정)
ACCEL=['ACCELERATING','DECELERATING','CONSTANT','STOPPED']
STEER=['LEFT','STRAIGHT','RIGHT']
FLOW_SIZE=(160,90)


In [ ]:
def load_frames(path):
    cap=cv2.VideoCapture(str(path)); out=[]
    while True:
        ok,bgr=cap.read()
        if not ok: break
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    if not out: raise ValueError(f'cannot decode: {path}')
    return out

def sample_frames(frames,n=8):
    idx=np.linspace(0,len(frames)-1,min(n,len(frames))).round().astype(int)
    return [frames[i] for i in idx]

# ---- Stage1(실제 사용): MViT 입력용 클립 ----
def _crop_tensor(rgb,size=224):
    h,w=rgb.shape[:2]; scale=size/min(h,w)
    nh,nw=max(size,round(h*scale)),max(size,round(w*scale))
    rgb=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    y,x=(nh-size)//2,(nw-size)//2
    return torch.from_numpy(rgb[y:y+size,x:x+size].copy()).permute(2,0,1).float()/255

def _clip(path,n=16,center=None):
    frames=load_frames(path); total=len(frames)
    if center is None: idx=np.linspace(0,total-1,n).round().astype(int)
    else: idx=np.clip(center-n//2+np.arange(n),0,total-1)
    x=torch.stack([_crop_tensor(frames[int(i)]) for i in idx],1)
    return x,total

# ---- Stage1(실험용, 미사용 - fit_stage1_experimental_handcrafted 전용): 재녹화 시뮬레이션 ----
def _moire_overlay(frame,rng):
    h,w=frame.shape[:2]; freq=rng.uniform(0.15,0.4); phase=rng.uniform(0,np.pi)
    yy,xx=np.mgrid[0:h,0:w]; grid=0.5+0.5*np.sin(freq*(xx+yy)+phase)
    strength=rng.uniform(6,18)
    out=frame.astype(np.float32)+(grid[...,None]-0.5)*strength
    return np.clip(out,0,255).astype(np.uint8)

def _flicker_stack(frames,rng):
    period=rng.uniform(3.5,9.0); amp=rng.uniform(0.06,0.16)
    return [np.clip(f.astype(np.float32)*(1+amp*np.sin(2*np.pi*i/period)),0,255).astype(np.uint8)
            for i,f in enumerate(frames)]

def _double_compress(frame,rng):
    quality=rng.randint(25,55); bgr=cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)
    ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
    bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

def _add_border(frame,rng):
    h,w=frame.shape[:2]; b=int(min(h,w)*rng.uniform(0.02,0.06)); out=frame.copy()
    out[:b]=out[-b:]=out[:,:b]=out[:,-b:]=0
    return out

def simulate_rerecording(frames,seed):
    rng=random.Random(seed)
    frames=_flicker_stack(frames,rng)
    frames=[_moire_overlay(f,rng) for f in frames]
    frames=[_double_compress(f,rng) for f in frames]
    frames=[_add_border(f,rng) for f in frames]
    return frames

def benign_augment(frames,seed):
    rng=random.Random(seed); gain=rng.uniform(0.85,1.15); quality=rng.randint(75,95); out=[]
    for f in frames:
        bright=np.clip(f.astype(np.float32)*gain,0,255).astype(np.uint8)
        bgr=cv2.cvtColor(bright,cv2.COLOR_RGB2BGR)
        ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
        bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    return out

def _fft_high_freq_ratio(gray):
    f=np.fft.fftshift(np.fft.fft2(gray.astype(np.float32))); mag=np.abs(f)
    h,w=gray.shape; cy,cx=h//2,w//2
    yy,xx=np.mgrid[0:h,0:w]; r=np.sqrt((yy-cy)**2+(xx-cx)**2)
    radius=min(h,w)*0.15
    return float(mag[r>radius].sum()/(mag.sum()+1e-6))

def _blockiness(gray):
    g=gray.astype(np.float32); h,w=g.shape; h,w=h-h%8,w-w%8; g=g[:h,:w]
    boundary=np.abs(np.diff(g[:,7:w:8],axis=1)).mean() if w>8 else 0.0
    boundary+=np.abs(np.diff(g[7:h:8,:],axis=0)).mean() if h>8 else 0.0
    interior=np.abs(np.diff(g,axis=1)).mean()+np.abs(np.diff(g,axis=0)).mean()
    return float(boundary/(interior+1e-6))

def extract_features(frames):
    frames=sample_frames(frames,8)
    grays=[cv2.cvtColor(f,cv2.COLOR_RGB2GRAY) for f in frames]
    resized=[cv2.resize(g,(256,256)) for g in grays]
    fft_ratio=float(np.mean([_fft_high_freq_ratio(g) for g in resized]))
    brightness=np.array([g.mean() for g in grays],dtype=np.float32)
    flicker_std=float(brightness.std()/(brightness.mean()+1e-6))
    border=float(np.mean([np.concatenate([g[:4].ravel(),g[-4:].ravel(),g[:,:4].ravel(),g[:,-4:].ravel()]).mean() for g in grays]))
    blur=float(np.mean([cv2.Laplacian(g,cv2.CV_64F).var() for g in resized]))
    block=float(np.mean([_blockiness(g) for g in resized]))
    return np.array([fft_ratio,flicker_std,border,blur,block],dtype=np.float32)


In [ ]:
# ---- Stage1(실제 사용): MViTv2-S (팀 베이스라인 원안, 0.53397로 검증됨) ----
class Stage1MViT(nn.Module):
    def __init__(self):
        super().__init__(); self.net=mvit_v2_s(weights=None)
        self.net.head[1]=nn.Linear(self.net.head[1].in_features,2)
    def forward(self,x): return self.net(x)

# ---- Stage2: COCO 탐지기 + 재정렬기 ----
def load_detector():
    weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
    model=fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights); model.eval()
    return model,weights.transforms(),weights.meta['categories']

@torch.inference_mode()
def detect_all_vehicles(model,transform,categories,frame,score_thr=0.2):
    x=transform(torch.from_numpy(frame).permute(2,0,1))
    out=model([x])[0]; candidates=[]
    for box,label,score in zip(out['boxes'],out['labels'],out['scores']):
        if score<score_thr or categories[label] not in VEHICLE_CLASSES: continue
        x0,y0,x1,y1=box.tolist(); candidates.append((x0,y0,x1,y1,float(score)))
    return candidates

def _rerank_features(c,w,h):
    x0,y0,x1,y1,score=c[:5]
    cx,cy=(x0+x1)/2/w,(y0+y1)/2/h; bw,bh=(x1-x0)/w,(y1-y0)/h
    return [cx,cy,bw,bh,score,bw/(bh+1e-6)]

def detect_vehicles(model,transform,categories,frame,reranker=None):
    candidates=detect_all_vehicles(model,transform,categories,frame,score_thr=SCORE_THR)
    if not candidates: return None
    if reranker is not None:
        net,mean,std=reranker; h,w=frame.shape[:2]
        feats=torch.tensor([_rerank_features(c,w,h) for c in candidates],dtype=torch.float32)
        with torch.inference_mode():
            scores=net((feats-mean)/std).squeeze(-1)
        return candidates[int(scores.argmax())]
    return max(candidates,key=lambda c:c[4]*(c[2]-c[0])*(c[3]-c[1]))

def motion_energy(frames):
    grays=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),(320,180)) for f in frames]
    diffs=[cv2.absdiff(grays[i],grays[i-1]).mean() for i in range(1,len(grays))]
    return np.array([diffs[0]]+diffs,dtype=np.float32)

def find_collision_frame(frames):
    energy=motion_energy(frames); margin=max(1,len(energy)//10)
    window=energy[margin:-margin] if len(energy)>2*margin else energy
    return int(np.argmax(window))+margin

def find_entry_and_scene(model,transform,categories,frames,collision_frame,reranker=None):
    h,w=frames[0].shape[:2]
    window_lo=max(0,collision_frame-30); window_hi=min(len(frames)-1,collision_frame+5)
    detections={}; entry_frame,entry_side=None,None
    for t in range(window_lo,window_hi+1):
        det=detect_vehicles(model,transform,categories,frames[t],reranker=reranker)
        if det is None: continue
        detections[t]=det
        x0,y0,x1,y1,score=det; area_frac=(x1-x0)*(y1-y0)/(w*h)
        if entry_frame is None and t<=collision_frame and area_frac>0.03:
            entry_frame=t; entry_side='LEFT' if (x0+x1)/2<w/2 else 'RIGHT'
    if entry_frame is None: entry_frame,entry_side=collision_frame,'RIGHT'
    box_at_collision=None
    if detections:
        nearest_t=min(detections,key=lambda t:abs(t-collision_frame)); box_at_collision=detections[nearest_t]
    evasion_space=0
    if box_at_collision is not None:
        x0,y0,x1,y1,score=box_at_collision
        evasion_space=int(max(x0,w-x1)>0.15*w)
    return entry_frame,entry_side,evasion_space

# ---- Stage3: optical flow (학습 없음, 임계값만 보정) ----
def compute_flow_series(frames):
    small=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),FLOW_SIZE) for f in frames]
    n=len(small)//2; w,h=FLOW_SIZE
    road=slice(int(h*0.55),h); horizon=slice(int(h*0.25),int(h*0.55))
    speed=np.zeros(n,dtype=np.float32); steer=np.zeros(n,dtype=np.float32)
    for t in range(n):
        i0=min(2*t,len(small)-3); i1=i0+2
        flow=cv2.calcOpticalFlowFarneback(small[i0],small[i1],None,0.5,2,15,3,5,1.2,0)
        mag=np.sqrt(flow[...,0]**2+flow[...,1]**2)
        speed[t]=float(np.median(mag[road])); steer[t]=float(np.median(flow[horizon,:,0]))
    return speed,steer

def _smooth(x,k=3):
    if len(x)<2*k+1: return x
    return np.convolve(x,np.ones(2*k+1)/(2*k+1),mode='same')

def classify(speed,steer,stopped_thr,accel_eps,steer_thr):
    # steer 스무딩 추가했다가(LOVO 0.670->0.710) 실제 점수가 0.521->0.48->0.47로
    # 계속 떨어져서 원복. Macro-F1 공식(팀 이슈 #3: 0.7*accel+0.3*steer, STOPPED 제외)
    # 으로 다시 그리드서치해도 스무딩 여부와 무관하게 같은 임계값이 최적이라, 스무딩
    # 자체의 문제라기보다 "공개 5비디오 로컬검증이 실제 숨은 평가셋과 거의 무관하다"는
    # 구조적 한계로 보임(팀장님도 반대방향 동일 현상 - 이슈 #10). 실측 검증된 상태로 복귀.
    speed_s=_smooth(speed); n=len(speed_s); accel_out,steer_out=[],[]
    for t in range(n):
        if speed_s[t]<stopped_thr: accel_out.append('STOPPED')
        else:
            lo,hi=max(0,t-3),min(n,t+4)
            slope=speed_s[hi-1]-speed_s[lo] if hi-1>lo else 0.0
            accel_out.append('ACCELERATING' if slope>accel_eps else 'DECELERATING' if slope<-accel_eps else 'CONSTANT')
        s=steer[t]
        # 좌회전->배경이 화면에서 오른쪽으로 흐름(flow_x 양수). AIHub 실측 자이로+실제 프레임으로 검증된 부호.
        steer_out.append('LEFT' if s>steer_thr else 'RIGHT' if s<-steer_thr else 'STRAIGHT')
    return accel_out,steer_out

In [ ]:
def _video_bpp(path):
    """size(bytes)*8 / (frame_count*width*height) - 해상도/길이 무관 압축난이도 지표.
    공개 10샘플에서 원본 vs 재녹화가 깨끗하게 갈림(LOO 9~10/10) - 단, DACON이
    재녹화 예제는 "실제 재촬영 아닌 파생 예제"라 명시해서 실제 평가셋에 안 통할
    위험 있음(인코더/코덱 차이가 DACON 예제 생성 파이프라인 특성일 수 있음).
    그래도 MViT(LOO 0/10, 페어 암기로 역방향)보다 훨씬 강한 로컬 신호라 공격적으로
    채택 - MViT를 보조(가중치 0.25)로만 섞어서 한쪽에만 전부 걸지는 않음."""
    cap=cv2.VideoCapture(str(path))
    nframes=cap.get(cv2.CAP_PROP_FRAME_COUNT); w=cap.get(cv2.CAP_PROP_FRAME_WIDTH); h=cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    if nframes<=0 or w<=0 or h<=0: return 0.0
    return Path(path).stat().st_size*8/(nframes*w*h)

def fit_stage1():
    """팀 베이스라인 원안(MViTv2-S, ImageNet 가중치 없이 5+5 공개샘플로 1 epoch 학습)
    + bpp(비트레이트) 신호를 0.75:0.25로 블렌드(공격적 시도, 실제 검증 전).
    MViT 단독 실제 제출 0.438(팀 베이스라인 0.53397에도 못 미침, 재현성도 의심)이라
    bpp 신호를 주신호로 채택 - 풀핏 LOO 검증상 10/10, MViT는 0/10(페어 암기)."""
    out=MODEL/'stage1'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage1/labels.csv')
    model=Stage1MViT().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),1e-4)
    model.train()
    for r in df.sample(frac=1,random_state=20260825).itertuples():
        x,_=_clip(DATA/'stage1'/r.path,16); x=(x-S1_MEAN)/S1_STD
        y=torch.tensor([0 if r.label=='ORIGINAL' else 1],device=DEVICE)
        loss=nn.functional.cross_entropy(model(x[None].to(DEVICE)),y)
        opt.zero_grad(); loss.backward(); opt.step()

    bpps={r.ID:_video_bpp(DATA/'stage1'/r.path) for r in df.itertuples()}
    o_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='ORIGINAL']
    r_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='RERECORDED']
    bpp_thr=(max(o_vals)+min(r_vals))/2; bpp_scale=max((min(r_vals)-max(o_vals))/2,1e-6)
    print(f'  stage1 bpp_thr={bpp_thr:.5f} bpp_scale={bpp_scale:.5f} (원본 bpp<={max(o_vals):.4f}, 재녹화 bpp>={min(r_vals):.4f})')

    torch.save({'model':model.net.state_dict(),'size':224,'frames':16,
                'bpp_thr':bpp_thr,'bpp_scale':bpp_scale,'bpp_weight':0.75}, out/'best.pt')

def fit_stage1_experimental_handcrafted():
    """실험용(기본 실행 대상 아님) - 합성 재녹화+benign 증강으로 만든 데이터로 5개 핸드크래프트
    피처+로지스틱회귀 학습. 로컬 합성데이터 정확도는 95~98%였지만 실제 제출 0.316으로
    베이스라인(0.53397)보다 나빴다 - 합성 artefact 과적합으로 추정. 실제 재녹화 데이터
    확보 전까지는 model/stage1/best.pt를 덮어쓰지 않도록 별도 경로에 저장한다."""
    out=MODEL/'stage1_experimental'; out.mkdir(parents=True,exist_ok=True)
    originals=sorted((DATA/'stage1/original').glob('*.mp4'))+sorted((DATA/'stage2/videos').glob('*.mp4'))+sorted((DATA/'stage3/videos').glob('*.mp4'))
    rerecorded=sorted((DATA/'stage1/rerecorded').glob('*.mp4'))
    X,y=[],[]
    for path in originals:
        frames=sample_frames(load_frames(path),8)
        X.append(extract_features(frames)); y.append(0)
        for v in range(3):
            X.append(extract_features(benign_augment(frames,seed=hash((path.name,v))&0xFFFF))); y.append(0)
            X.append(extract_features(simulate_rerecording(frames,seed=hash((path.name,v,'r'))&0xFFFF))); y.append(1)
    for path in rerecorded:
        X.append(extract_features(load_frames(path))); y.append(1)
    X,y=np.stack(X),np.array(y,dtype=np.float32)

    mean,std=X.mean(0),X.std(0)+1e-6
    Xn=torch.tensor((X-mean)/std,dtype=torch.float32); yt=torch.tensor(y)
    weight=torch.zeros(X.shape[1],requires_grad=True); bias=torch.zeros(1,requires_grad=True)
    opt=torch.optim.Adam([weight,bias],lr=0.1)
    for _ in range(300):
        loss=nn.functional.binary_cross_entropy_with_logits(Xn@weight+bias,yt)
        opt.zero_grad(); loss.backward(); opt.step()
    acc=float(((Xn@weight+bias>0).float()==yt).float().mean())
    print(f'  stage1(실험용) train acc: {acc:.3f} (합성 데이터 기준 - 실제 0.316으로 검증됨, 참고용)')
    torch.save({'weight':weight.detach(),'bias':bias.detach(),'feat_mean':mean,'feat_std':std}, out/'best.pt')

def fit_stage2():
    """COCO 사전학습 탐지기는 그대로 저장(파인튜닝 아님). 재정렬기(reranker.pt)는 AIHub
    실라벨(597, 차대차 카테고리)로 별도 학습한 산출물 - 공개 5샘플엔 그런 라벨이 없어서
    여기서 재현 불가. 이미 model/stage2/reranker.pt가 있으면(로컬에서 미리 학습) 그대로 두고,
    없으면 score*area 폴백으로 동작 (detect_vehicles 참고)."""
    out=MODEL/'stage2'; out.mkdir(parents=True,exist_ok=True)
    model,_,_=load_detector()
    torch.save(model.state_dict(), out/'detector.pth')
    reranker_path=out/'reranker.pt'
    print(f"  reranker.pt {'존재 - 유지' if reranker_path.exists() else '없음 - score*area 폴백으로 동작'}")

def fit_stage3():
    """optical flow는 학습이 아니라 임계값 그리드서치. 공개 라벨 50개(6초 간격) 기준.
    LOVO(leave-one-video-out) 교차검증으로 steer 스무딩 추가가 실제로 낫다는 것을 확인함
    (0.670->0.710). 그리드를 10x10x10으로 넓혀봤다가 실제 제출 점수가 0.521->0.48로
    떨어져서(로컬 LOVO에서도 그리드만 넓힌 건 0.710->0.700으로 오히려 나빴음 - 5개
    비디오짜리 초소량 데이터라 그리드를 촘촘히 훑을수록 과적합 위험) 그리드는 원래
    6x6x6으로 되돌리고 steer 스무딩만 유지한다."""
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    labels=pd.read_csv(DATA/'stage3/labels.csv')
    cache={}
    for vid_id,group in labels.groupby('ID'):
        frames=load_frames(DATA/'stage3/videos'/f'{vid_id}.mp4')
        cache[vid_id]=(compute_flow_series(frames),group)

    best=None
    grid=itertools.product(np.linspace(0.1,1.5,6), np.linspace(0.02,0.3,6), np.linspace(0.1,1.0,6))
    for stopped_thr,accel_eps,steer_thr in grid:
        correct=total=0
        for vid_id,((speed,steer),group) in cache.items():
            accel_pred,steer_pred=classify(speed,steer,stopped_thr,accel_eps,steer_thr)
            for row in group.itertuples():
                idx=min(row.sample_index,len(accel_pred)-1); total+=2
                correct+=accel_pred[idx]==row.accel_label; correct+=steer_pred[idx]==row.steer_label
        acc=correct/total
        if best is None or acc>best[0]: best=(acc,stopped_thr,accel_eps,steer_thr)
    acc,stopped_thr,accel_eps,steer_thr=best
    print(f'  stage3 calibrated acc: {acc:.3f} (stopped_thr={stopped_thr:.3f}, accel_eps={accel_eps:.3f}, steer_thr={steer_thr:.3f})')
    torch.save({'stopped_thr':stopped_thr,'accel_eps':accel_eps,'steer_thr':steer_thr}, out/'best.pt')

## 2. Stage 1·2·3 학습

In [ ]:
print('device:',DEVICE)
fit_stage1(); print('Stage 1 완료')
fit_stage2(); print('Stage 2 완료')
fit_stage3(); print('Stage 3 완료')

In [ ]:
for p in sorted((ROOT/'model').rglob('*')):
    if p.is_file(): print(p.relative_to(ROOT),f'{p.stat().st_size/1024**2:.1f} MB')